# Flipkart Gridlock 2.0: V12 Stacked Triad Architecture
## Level-2 Meta-Modeling (Ridge Regression)

**Architectural Paradigm:**
This pipeline implements a Level-2 Stacking strategy. It generates independent Out-of-Fold (OOF) predictions from three topologically distinct gradient boosting engines (Leaf-wise, Depth-wise, and Symmetrical). These predictions form a secondary feature matrix, which is processed by a dynamically cross-validated Ridge Regressor to autonomously learn localized model trust weights, optimizing the ensemble for the 93.0 R² threshold.

[Environment & Matrix Ingestion]

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import pygeohash as pgh
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
np.random.seed(42)

data_paths = [".", "data/raw", "../../data/raw"]
base_path = next((path for path in data_paths if os.path.exists(os.path.join(path, "train.csv"))), None)

raw_train = pd.read_csv(os.path.join(base_path, "train.csv"))
raw_test = pd.read_csv(os.path.join(base_path, "test.csv"))

y_train = raw_train['demand'].values
submission_index = raw_test['Index'].values

X_train_base = raw_train.drop(columns=['demand'], errors='ignore')
X_test_base = raw_test.drop(columns=['Index'], errors='ignore')

print("System initialized. Level-1 base matrices loaded.")

System initialized. Level-1 base matrices loaded.


[Autoregressive Memory & Spatial Topology]

In [2]:
def engineer_master_features(source_df, target_df):
    df_f = target_df.copy()
    
    # 1. Autoregressive Memory Injector
    lag_24 = source_df[['geohash', 'day', 'timestamp', 'demand']].copy()
    lag_24['day'] += 1
    lag_24.rename(columns={'demand': 'lag_24h'}, inplace=True)
    
    lag_48 = source_df[['geohash', 'day', 'timestamp', 'demand']].copy()
    lag_48['day'] += 2
    lag_48.rename(columns={'demand': 'lag_48h'}, inplace=True)
    
    df_f = df_f.merge(lag_24, on=['geohash', 'day', 'timestamp'], how='left')
    df_f = df_f.merge(lag_48, on=['geohash', 'day', 'timestamp'], how='left')
    df_f[['lag_24h', 'lag_48h']] = df_f[['lag_24h', 'lag_48h']].fillna(0.0)
    
    # 2. Temporal Kinematics
    t_split = df_f['timestamp'].str.split(':', expand=True).astype(int)
    df_f['ts_minutes'] = t_split[0] * 60 + t_split[1]
    df_f['hour'] = t_split[0]
    df_f['time_slot_15m'] = df_f['ts_minutes'] // 15
    
    df_f['hour_sin'] = np.sin(2 * np.pi * df_f['hour'] / 24.0)
    df_f['hour_cos'] = np.cos(2 * np.pi * df_f['hour'] / 24.0)
    df_f['min_sin'] = np.sin(2 * np.pi * df_f['ts_minutes'] / 1440.0)
    df_f['min_cos'] = np.cos(2 * np.pi * df_f['ts_minutes'] / 1440.0)
    
    # 3. High-Resolution Interaction Key
    df_f['geo_time_interaction'] = df_f['geohash'].astype(str) + "_" + df_f['time_slot_15m'].astype(str)
    
    # 4. Traffic Physics Enhancements
    df_f['is_rush_hour'] = df_f['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
    
    # 5. Spatial Geometry Decoding
    coords = df_f['geohash'].apply(lambda x: pgh.decode(x) if isinstance(x, str) else (np.nan, np.nan))
    df_f['lat'] = coords.apply(lambda x: x[0])
    df_f['lon'] = coords.apply(lambda x: x[1])
    
    # 6. Fallback Imputations
    df_f['Temperature'] = df_f['Temperature'].fillna(df_f['Temperature'].median())
    for col in ['Weather', 'RoadType', 'LargeVehicles', 'Landmarks']:
        if col in df_f.columns:
            df_f[col] = df_f[col].fillna('Unknown')
            
    return df_f

X_train_fe = engineer_master_features(raw_train, X_train_base)
X_test_fe = engineer_master_features(raw_train, X_test_base)

# 25-Hub Macro Clustering
g_coords = pd.concat([X_train_fe[['lat', 'lon']], X_test_fe[['lat', 'lon']]]).dropna()
kmeans_25 = KMeans(n_clusters=25, random_state=42, n_init=10).fit(g_coords)
X_train_fe['cluster_25'] = kmeans_25.predict(X_train_fe[['lat', 'lon']].fillna(0))
X_test_fe['cluster_25'] = kmeans_25.predict(X_test_fe[['lat', 'lon']].fillna(0))

print("Feature space engineered: Kinematics, Lags, and Clusters mapped.")

Feature space engineered: Kinematics, Lags, and Clusters mapped.


[Out-Of-Fold Target Encoding (Leak-Free Protocol)]

In [3]:
def apply_oof_encoding(tr_df, te_df, tgt, col, folds=5):
    kf = KFold(n_splits=folds, shuffle=True, random_state=42)
    tr_enc = np.zeros(len(tr_df))
    
    tmp_tr = tr_df[[col]].copy()
    tmp_tr['tgt'] = tgt
    g_mean = tgt.mean()
    
    # OOF training execution
    for tr_idx, val_idx in kf.split(tmp_tr):
        f_map = tmp_tr.iloc[tr_idx].groupby(col)['tgt'].mean()
        tr_enc[val_idx] = tmp_tr.iloc[val_idx][col].map(f_map).fillna(g_mean).values
        
    # Global mapping for evaluation set
    te_map = tmp_tr.groupby(col)['tgt'].mean()
    te_enc = te_df[col].map(te_map).fillna(g_mean).values
        
    return tr_enc, te_enc

# Target Encode Base Geohash and the High-Cardinality Interaction Key
X_train_fe['TE_geohash'], X_test_fe['TE_geohash'] = apply_oof_encoding(X_train_fe, X_test_fe, y_train, 'geohash')
X_train_fe['TE_geo_time'], X_test_fe['TE_geo_time'] = apply_oof_encoding(X_train_fe, X_test_fe, y_train, 'geo_time_interaction')

# Standard Label Encoding for residual string categories
cat_features = ['geohash', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']
for c in cat_features:
    le = LabelEncoder()
    le.fit(X_train_fe[c].astype(str).tolist() + X_test_fe[c].astype(str).tolist())
    X_train_fe[c] = le.transform(X_train_fe[c].astype(str))
    X_test_fe[c] = le.transform(X_test_fe[c].astype(str))

# Format Final Matrix
drop_columns = ['timestamp', 'geo_time_interaction', 'Index']
features = [c for c in X_train_fe.columns if c not in drop_columns]

X = X_train_fe[features].values
X_test = X_test_fe[features].values
X_test = np.nan_to_num(X_test, nan=0.0) # Safety net for missing float boundaries

print(f"Matrix formatting completed. Level-1 Input Shape: {X.shape}")

Matrix formatting completed. Level-1 Input Shape: (77299, 23)


[ Level-1 Base Engine Training (The Triad) ]

In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Algorithm bounds configured to heavily penalize over-segmentation
lgb_p = {'objective': 'regression', 'metric': 'rmse', 'learning_rate': 0.03, 'max_depth': 8, 'num_leaves': 128, 'min_child_samples': 20, 'verbose': -1, 'random_state': 42, 'n_jobs': -1}
xgb_p = {'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'learning_rate': 0.03, 'max_depth': 7, 'random_state': 42, 'n_jobs': -1}
cat_p = {'iterations': 2500, 'learning_rate': 0.03, 'depth': 8, 'eval_metric': 'RMSE', 'verbose': 0, 'random_seed': 42}

oof_lgb, test_lgb = np.zeros(len(X)), np.zeros(len(X_test))
oof_xgb, test_xgb = np.zeros(len(X)), np.zeros(len(X_test))
oof_cat, test_cat = np.zeros(len(X)), np.zeros(len(X_test))

print("Initiating Level-1 Triad Training Protocol...")

for fold, (t_idx, v_idx) in enumerate(kf.split(X)):
    X_tr, y_tr = X[t_idx], y_train[t_idx]
    X_va, y_va = X[v_idx], y_train[v_idx]
    
    # 1. Leaf-Wise Evaluation
    m_lgb = lgb.train(lgb_p, lgb.Dataset(X_tr, y_tr), 2500, valid_sets=[lgb.Dataset(X_va, y_va)], callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_lgb[v_idx] = m_lgb.predict(X_va)
    test_lgb += m_lgb.predict(X_test) / 5
    
    # 2. Depth-Wise Evaluation
    m_xgb = xgb.train(xgb_p, xgb.DMatrix(X_tr, y_tr), 2500, evals=[(xgb.DMatrix(X_va, y_va), 'val')], early_stopping_rounds=100, verbose_eval=False)
    oof_xgb[v_idx] = m_xgb.predict(xgb.DMatrix(X_va))
    test_xgb += m_xgb.predict(xgb.DMatrix(X_test)) / 5
    
    # 3. Symmetrical Oblivious Evaluation
    m_cat = CatBoostRegressor(**cat_p).fit(X_tr, y_tr, eval_set=(X_va, y_va))
    oof_cat[v_idx] = m_cat.predict(X_va)
    test_cat += m_cat.predict(X_test) / 5
    
    print(f"Level-1 Fold {fold+1} validation finalized.")

Initiating Level-1 Triad Training Protocol...
Level-1 Fold 1 validation finalized.
Level-1 Fold 2 validation finalized.
Level-1 Fold 3 validation finalized.
Level-1 Fold 4 validation finalized.
Level-1 Fold 5 validation finalized.


[Level-2 Meta-Model Stacking (Ridge Regressor)]

In [5]:
print("Initiating Level-2 Meta-Model Stacking...")

# 1. Construct the Level-1 Feature Matrix from Out-Of-Fold predictions
X_meta_train = np.column_stack([oof_lgb, oof_xgb, oof_cat])
X_meta_test = np.column_stack([test_lgb, test_xgb, test_cat])

# 2. Train a highly regularized Meta-Model (Ridge Regression with built-in CV)
# This model learns the dynamic spatial weaknesses of the three base algorithms
meta_model = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0], cv=5)
meta_model.fit(X_meta_train, y_train)

# 3. Generate Final Stacked Predictions
oof_stacked = meta_model.predict(X_meta_train)
test_stacked = meta_model.predict(X_meta_test)

# 4. Enforce Physical Capacity Boundaries [0.0, 1.0]
oof_stacked = np.clip(oof_stacked, 0.0, 1.0)
final_test_predictions = np.clip(test_stacked, 0.0, 1.0)

# Evaluate the final Level-2 Matrix
final_stacked_r2 = max(0, 100 * r2_score(y_train, oof_stacked))

# 5. Pipeline Export
submission_payload = pd.DataFrame({
    'Index': submission_index,
    'demand': final_test_predictions
})
submission_filename = "submission_v12.csv"
submission_payload.to_csv(submission_filename, index=False)

print("\n==================================================")
print("PIPELINE EXECUTION COMPLETE (V12 STACKED)")
print("==================================================")
print(f"Meta-Model Selected Regularization (Alpha): {meta_model.alpha_}")
print(f"Base Engine Influence Array: {meta_model.coef_}")
print(f"Terminal Level-2 Stacked R2 Score: {final_stacked_r2:.4f}")
print(f"Output Matrix Generated: {submission_filename}")

Initiating Level-2 Meta-Model Stacking...

PIPELINE EXECUTION COMPLETE (V12 STACKED)
Meta-Model Selected Regularization (Alpha): 1.0
Base Engine Influence Array: [0.22315542 0.41160147 0.37063564]
Terminal Level-2 Stacked R2 Score: 95.5940
Output Matrix Generated: submission_v12.csv
